# DUP — Current Batch Load and Anomaly Detection

| | |
|---|---|
| **Notebook** | `DUP_current_batch_load.ipynb` |
| **Description** | Processes current Duplicate QC batch data and evaluates analytical precision against the historical reference model. |
| **Author** | Rob Clark |
| **Created** | September 2026 / Semester 2 |
| **QC Type** | Duplicate |

## Change History

| Date | Author | Description |
|---|---|---|
| 2026-09-07 | R.Clark | (T32-144) Created Current Batch Load notebook and established workflow for evaluating Duplicate QC observations against the persisted historical reference model. |

## Purpose

This notebook processes a current batch of Duplicate QC observations and evaluates their analytical behaviour against the persisted historical Duplicate reference model.

The workflow will:

1. Load and validate current batch data.
2. Select and pair Duplicate QC observations.
3. Calculate precision features including **Mean Concentration** and **RPD**.
4. Load the persisted historical Duplicate reference model.
5. Map current observations to analyte-specific historical reference values.
6. Compare current RPD against historical **Q95** and **Limiting Repeatability** thresholds.
7. Identify and rank potential anomalies for review.

The current batch is **not used to fit or modify the historical model**. It is evaluated against the previously established historical reference.


In [1]:
# =============================================================================
# Imports and configuration
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------------------------------------------------------
# Repository paths
# -----------------------------------------------------------------------------

DATA_DIR = Path("../data")
RAW_DATA_DIR = DATA_DIR / "raw"
REFERENCE_DATA_DIR = DATA_DIR / "reference"

CURRENT_BATCH_FILE = RAW_DATA_DIR / "ResultSet.csv"
REFERENCE_MODEL_FILE = (
    REFERENCE_DATA_DIR / "DUP_historical_reference_model.csv"
)


# -----------------------------------------------------------------------------
# Confirm configuration
# -----------------------------------------------------------------------------

print(f"Current batch source : {CURRENT_BATCH_FILE}")
print(f"Reference model      : {REFERENCE_MODEL_FILE}")


Current batch source : ..\data\raw\ResultSet.csv
Reference model      : ..\data\reference\DUP_historical_reference_model.csv


In [2]:
# =============================================================================
# Load current batch data
# =============================================================================

df = pd.read_csv(
    CURRENT_BATCH_FILE,
    dtype={
        "INSTRUMENT_ID": "string"
    }
)

print(f"Rows loaded: {len(df):,}")
print(f"Columns: {len(df.columns)}")

df.head()


Rows loaded: 99,999
Columns: 27


,ANALYTICAL_TYPE,STD_LOT_CODE,STD_CODE,JOB_CODE,NUMERIC_FINAL_VALUE,ANALYSED_DATE,SCHEME_CODE,ANALYTE_CODE,STANDARD_STATUS,PRECISION_STATUS,...,INTERNAL_MAX_WARNING_INCLUSIVE,LIM_REP_VALUE,STAT_DL_VALUE,LIM_REP_DUP_VALUE,STAT_DL_DUP_VALUE,INTERNAL_TARGET_VALUE,PARENT_NUMERIC_FINAL_VALUE,UNIT_CODE,SPECIFICATION_CODE,INSTRUMENT_ID
0,Standard,OREAS_905,OREAS_905,TSV_LB0016663011,35.927171,2021-11-22 16:37:52.000000,GE_IMS40Q12,PB,UpperFailure,NaN,...,Y,10,1.25,15.0,1.25,30.40,NaN,MG_KG,OREAS_905,<NA>
1,Standard,OREAS_905,OREAS_905,TSV_LB0016663011,55.639610,2021-11-22 16:37:52.000000,GE_IMS40Q12,PB,UpperFailure,NaN,...,Y,10,1.25,15.0,1.25,30.40,NaN,MG_KG,OREAS_905,<NA>
2,Replicate,Sample,Sample,TSV_LB0016663011,1492.268392,2021-11-22 16:37:52.000000,GE_IMS40Q12,PB,NaN,Warning,...,NaN,10,1.25,15.0,1.25,NaN,1385.198107,MG_KG,NaN,<NA>
3,Replicate,Sample,Sample,TSV_LB0016663011,18.525452,2021-11-22 16:37:33.000000,GE_IMS40Q12,NI,NaN,Pass,...,NaN,10,5.00,15.0,5.00,NaN,16.334782,MG_KG,NaN,<NA>
4,Standard,OREAS_905,OREAS_905,TSV_LB0016663011,11.742601,2021-11-22 16:37:33.000000,GE_IMS40Q12,NI,UpperWarning,NaN,...,Y,10,5.00,15.0,5.00,9.54,NaN,MG_KG,OREAS_905,<NA>


In [3]:
# =============================================================================
# Select Duplicate QC observations
# =============================================================================

df_dup = (
    df[
        (df["ANALYTICAL_TYPE"] == "Duplicate")
        & (df["STD_LOT_CODE"] == "Sample")
        & (df["STD_CODE"] == "Sample")
    ]
    .copy()
)

print(f"Duplicate rows selected: {len(df_dup):,}")
print(f"Jobs represented: {df_dup['JOB_CODE'].nunique():,}")
print(f"Analytes represented: {df_dup['ANALYTE_CODE'].nunique():,}")
print(f"Schemes represented: {df_dup['SCHEME_CODE'].nunique():,}")

print("\nDuplicate rows by scheme:")
display(
    df_dup["SCHEME_CODE"]
    .value_counts(dropna=False)
    .to_frame("ROWS")
)

df_dup.head()


Duplicate rows selected: 5,075
Jobs represented: 47
Analytes represented: 41
Schemes represented: 1

Duplicate rows by scheme:


,ROWS
SCHEME_CODE,
GE_ICP40Q12,5075


,ANALYTICAL_TYPE,STD_LOT_CODE,STD_CODE,JOB_CODE,NUMERIC_FINAL_VALUE,ANALYSED_DATE,SCHEME_CODE,ANALYTE_CODE,STANDARD_STATUS,PRECISION_STATUS,...,INTERNAL_MAX_WARNING_INCLUSIVE,LIM_REP_VALUE,STAT_DL_VALUE,LIM_REP_DUP_VALUE,STAT_DL_DUP_VALUE,INTERNAL_TARGET_VALUE,PARENT_NUMERIC_FINAL_VALUE,UNIT_CODE,SPECIFICATION_CODE,INSTRUMENT_ID
19307,Duplicate,Sample,Sample,TSV_LB0007353557,2.185365,2021-05-10 11:39:33.000000,GE_ICP40Q12,TA,NaN,Pass,...,NaN,10,12.5,15.0,12.5,NaN,3.378640,MG_KG,NaN,<NA>
19308,Duplicate,Sample,Sample,TSV_LB0007353557,0.801886,2021-05-10 11:39:33.000000,GE_ICP40Q12,TA,NaN,Pass,...,NaN,10,12.5,15.0,12.5,NaN,-0.746270,MG_KG,NaN,<NA>
19309,Duplicate,Sample,Sample,TSV_LB0007353557,1.260273,2021-05-10 11:39:33.000000,GE_ICP40Q12,TA,NaN,Pass,...,NaN,10,12.5,15.0,12.5,NaN,1.879069,MG_KG,NaN,<NA>
19323,Duplicate,Sample,Sample,TSV_LB0007353557,-2.010870,2021-05-10 11:38:57.000000,GE_ICP40Q12,TA,NaN,Pass,...,NaN,10,12.5,15.0,12.5,NaN,-3.129188,MG_KG,NaN,<NA>
19324,Duplicate,Sample,Sample,TSV_LB0007353557,-3.709679,2021-05-10 11:38:57.000000,GE_ICP40Q12,TA,NaN,Pass,...,NaN,10,12.5,15.0,12.5,NaN,-2.185186,MG_KG,NaN,<NA>


In [4]:
# =============================================================================
# Build current Duplicate precision dataset
# =============================================================================

# Count Duplicate rows that cannot be linked to an analyte-specific
# historical reference model
missing_analyte_rows = (
    df_dup["ANALYTE_CODE"]
    .isna()
    .sum()
)

# Retain only Duplicate observations with a valid analyte code
current_dup_df = (
    df_dup[
        df_dup["ANALYTE_CODE"].notna()
    ][
        [
            "JOB_CODE",
            "ANALYSED_DATE",
            "ANALYTE_CODE",
            "SCHEME_CODE",
            "PARENT_NUMERIC_FINAL_VALUE",
            "NUMERIC_FINAL_VALUE",
            "STAT_DL_DUP_VALUE",
            "LIM_REP_DUP_VALUE",
            "PRECISION_STATUS",
            "UNIT_CODE"
        ]
    ]
    .copy()
)

# -----------------------------------------------------------------------------
# Derive precision features
# -----------------------------------------------------------------------------

# Absolute difference between original and Duplicate result
current_dup_df["ABS_DIFF"] = abs(
    current_dup_df["NUMERIC_FINAL_VALUE"]
    - current_dup_df["PARENT_NUMERIC_FINAL_VALUE"]
)

# Mean concentration of the paired results
current_dup_df["MEAN_CONC"] = (
    current_dup_df["NUMERIC_FINAL_VALUE"]
    + current_dup_df["PARENT_NUMERIC_FINAL_VALUE"]
) / 2

# Relative Percent Difference
current_dup_df["RPD"] = np.where(
    (
        current_dup_df["NUMERIC_FINAL_VALUE"]
        + current_dup_df["PARENT_NUMERIC_FINAL_VALUE"]
    ) != 0,
    (
        current_dup_df["ABS_DIFF"]
        / current_dup_df["MEAN_CONC"]
        * 100
    ),
    np.nan
)

# Round RPD for reporting consistency
current_dup_df["RPD"] = (
    current_dup_df["RPD"]
    .round(2)
)

# -----------------------------------------------------------------------------
# Reorder columns for downstream scoring
# -----------------------------------------------------------------------------

current_dup_df = current_dup_df[
    [
        "JOB_CODE",
        "ANALYSED_DATE",
        "ANALYTE_CODE",
        "SCHEME_CODE",
        "PARENT_NUMERIC_FINAL_VALUE",
        "NUMERIC_FINAL_VALUE",
        "MEAN_CONC",
        "ABS_DIFF",
        "RPD",
        "STAT_DL_DUP_VALUE",
        "LIM_REP_DUP_VALUE",
        "PRECISION_STATUS",
        "UNIT_CODE"
    ]
]

# -----------------------------------------------------------------------------
# Confirm prepared dataset
# -----------------------------------------------------------------------------

print(f"Duplicate observations available: {len(df_dup):,}")
print(f"Excluded missing ANALYTE_CODE:    {missing_analyte_rows:,}")
print(f"Current Duplicate observations:   {len(current_dup_df):,}")

print(
    f"Observations with paired parent value: "
    f"{current_dup_df['PARENT_NUMERIC_FINAL_VALUE'].notna().sum():,}"
)

print(
    f"Analytes represented: "
    f"{current_dup_df['ANALYTE_CODE'].nunique():,}"
)

current_dup_df.head(20)


Duplicate observations available: 5,075
Excluded missing ANALYTE_CODE:    136
Current Duplicate observations:   4,939
Observations with paired parent value: 4,939
Analytes represented: 41


,JOB_CODE,ANALYSED_DATE,ANALYTE_CODE,SCHEME_CODE,PARENT_NUMERIC_FINAL_VALUE,NUMERIC_FINAL_VALUE,MEAN_CONC,ABS_DIFF,RPD,STAT_DL_DUP_VALUE,LIM_REP_DUP_VALUE,PRECISION_STATUS,UNIT_CODE
19307,TSV_LB0007353557,2021-05-10 11:39:33.000000,TA,GE_ICP40Q12,3.378640,2.185365,2.782003,1.193275,42.89,12.500,15.0,Pass,MG_KG
19308,TSV_LB0007353557,2021-05-10 11:39:33.000000,TA,GE_ICP40Q12,-0.746270,0.801886,0.027808,1.548156,5567.32,12.500,15.0,Pass,MG_KG
19309,TSV_LB0007353557,2021-05-10 11:39:33.000000,TA,GE_ICP40Q12,1.879069,1.260273,1.569671,0.618796,39.42,12.500,15.0,Pass,MG_KG
19323,TSV_LB0007353557,2021-05-10 11:38:57.000000,TA,GE_ICP40Q12,-3.129188,-2.010870,-2.570029,1.118317,-43.51,12.500,15.0,Pass,MG_KG
19324,TSV_LB0007353557,2021-05-10 11:38:57.000000,TA,GE_ICP40Q12,-2.185186,-3.709679,-2.947432,1.524493,-51.72,12.500,15.0,Pass,MG_KG
19330,TSV_LB0007353557,2021-05-10 11:38:44.000000,TE,GE_ICP40Q12,-4.130237,5.205476,0.537619,9.335714,1736.49,25.000,15.0,Pass,MG_KG
19333,TSV_LB0007353557,2021-05-10 11:38:44.000000,TE,GE_ICP40Q12,2.379626,3.537630,2.958628,1.158004,39.14,25.000,15.0,Pass,MG_KG
19337,TSV_LB0007353557,2021-05-10 11:38:44.000000,TE,GE_ICP40Q12,2.995021,-5.698118,-1.351549,8.693140,-643.20,25.000,15.0,Pass,MG_KG
19339,TSV_LB0007353557,2021-05-10 11:38:44.000000,TE,GE_ICP40Q12,-1.864082,-2.273175,-2.068629,0.409093,-19.78,25.000,15.0,Pass,MG_KG
19343,TSV_LB0007353557,2021-05-10 11:38:44.000000,TE,GE_ICP40Q12,1.148321,1.956517,1.552419,0.808196,52.06,25.000,15.0,Pass,MG_KG


In [5]:
# =============================================================================
# Apply current Duplicate scoring eligibility rules
# =============================================================================

current_dup_score_df = (
    current_dup_df[
        current_dup_df["MEAN_CONC"].notna()
        & current_dup_df["RPD"].notna()
        & (current_dup_df["MEAN_CONC"] > 0)
    ]
    .copy()
)

print(f"Current Duplicate observations: {len(current_dup_df):,}")
print(f"Eligible for reference scoring: {len(current_dup_score_df):,}")
print(
    f"Excluded from scoring: "
    f"{len(current_dup_df) - len(current_dup_score_df):,}"
)

print(
    f"Analytes eligible for scoring: "
    f"{current_dup_score_df['ANALYTE_CODE'].nunique()}"
)

current_dup_score_df.head(20)


Current Duplicate observations: 4,939
Eligible for reference scoring: 4,437
Excluded from scoring: 502
Analytes eligible for scoring: 41


,JOB_CODE,ANALYSED_DATE,ANALYTE_CODE,SCHEME_CODE,PARENT_NUMERIC_FINAL_VALUE,NUMERIC_FINAL_VALUE,MEAN_CONC,ABS_DIFF,RPD,STAT_DL_DUP_VALUE,LIM_REP_DUP_VALUE,PRECISION_STATUS,UNIT_CODE
19307,TSV_LB0007353557,2021-05-10 11:39:33.000000,TA,GE_ICP40Q12,3.378640,2.185365,2.782003,1.193275,42.89,12.500,15.0,Pass,MG_KG
19308,TSV_LB0007353557,2021-05-10 11:39:33.000000,TA,GE_ICP40Q12,-0.746270,0.801886,0.027808,1.548156,5567.32,12.500,15.0,Pass,MG_KG
19309,TSV_LB0007353557,2021-05-10 11:39:33.000000,TA,GE_ICP40Q12,1.879069,1.260273,1.569671,0.618796,39.42,12.500,15.0,Pass,MG_KG
19330,TSV_LB0007353557,2021-05-10 11:38:44.000000,TE,GE_ICP40Q12,-4.130237,5.205476,0.537619,9.335714,1736.49,25.000,15.0,Pass,MG_KG
19333,TSV_LB0007353557,2021-05-10 11:38:44.000000,TE,GE_ICP40Q12,2.379626,3.537630,2.958628,1.158004,39.14,25.000,15.0,Pass,MG_KG
19343,TSV_LB0007353557,2021-05-10 11:38:44.000000,TE,GE_ICP40Q12,1.148321,1.956517,1.552419,0.808196,52.06,25.000,15.0,Pass,MG_KG
19358,TSV_LB0007353557,2021-05-10 11:34:57.000000,MO,GE_ICP40Q12,1.975463,1.891936,1.933700,0.083527,4.32,2.500,15.0,Pass,MG_KG
19361,TSV_LB0007353557,2021-05-10 11:34:57.000000,MO,GE_ICP40Q12,1.236364,0.896196,1.066280,0.340168,31.90,2.500,15.0,Pass,MG_KG
19365,TSV_LB0007353557,2021-05-10 11:34:57.000000,MO,GE_ICP40Q12,1.889768,1.536987,1.713377,0.352781,20.59,2.500,15.0,Pass,MG_KG
19366,TSV_LB0007353557,2021-05-10 11:34:57.000000,MO,GE_ICP40Q12,2.219903,5.282440,3.751172,3.062537,81.64,2.500,15.0,Warning,MG_KG


In [6]:
# =============================================================================
# Load persisted Duplicate historical reference model
# =============================================================================

REFERENCE_MODEL_FILE = "../data/reference/DUP_historical_reference_model.csv"

reference_model_df = pd.read_csv(
    REFERENCE_MODEL_FILE
)

print(f"Reference model rows: {len(reference_model_df):,}")
print(
    f"Reference analytes: "
    f"{reference_model_df['ANALYTE_CODE'].nunique():,}"
)

print("\nReference model columns:")
print(", ".join(reference_model_df.columns))

reference_model_df.head()


Reference model rows: 12,300
Reference analytes: 41

Reference model columns:
ANALYTE_CODE, LOG_MEAN_CONC, MEAN_CONC, Q25, Q50, Q75, Q95, STAT_DL_DUP_VALUE, LIM_REP_DUP_VALUE, ALLOWABLE_RPD


,ANALYTE_CODE,LOG_MEAN_CONC,MEAN_CONC,Q25,Q50,Q75,Q95,STAT_DL_DUP_VALUE,LIM_REP_DUP_VALUE,ALLOWABLE_RPD
0,AG,-2.246569,0.005668,1285.710000,1285.710006,2678.039991,2144.041928,1.25,15.0,22068.571428
1,AG,-2.236281,0.005804,1262.846778,1264.871479,2633.105281,2143.103373,1.25,15.0,21552.302101
2,AG,-2.225994,0.005943,1240.257721,1244.270097,2588.694894,2141.870919,1.25,15.0,21048.118528
3,AG,-2.215706,0.006085,1217.941185,1223.904482,2544.805718,2140.347581,1.25,15.0,20555.737782
4,AG,-2.205418,0.006231,1195.895526,1203.773254,2501.434639,2138.536369,1.25,15.0,20074.883564


In [7]:
# =============================================================================
# Validate current Duplicate analytes against historical reference model
# =============================================================================

current_analytes = set(
    current_dup_score_df["ANALYTE_CODE"].dropna().unique()
)

reference_analytes = set(
    reference_model_df["ANALYTE_CODE"].dropna().unique()
)

matched_analytes = current_analytes & reference_analytes
missing_analytes = current_analytes - reference_analytes
unused_reference_analytes = reference_analytes - current_analytes

print(f"Current analytes eligible for scoring: {len(current_analytes):,}")
print(f"Historical reference analytes:       {len(reference_analytes):,}")
print(f"Matched analytes:                    {len(matched_analytes):,}")
print(f"Missing historical reference:        {len(missing_analytes):,}")

if missing_analytes:
    print(
        "\nCurrent analytes without a historical reference model:"
    )
    print(", ".join(sorted(missing_analytes)))
else:
    print(
        "\nAll current Duplicate analytes have a historical reference model."
    )

if unused_reference_analytes:
    print(
        "\nHistorical analytes not represented in the current batch:"
    )
    print(", ".join(sorted(unused_reference_analytes)))
    

Current analytes eligible for scoring: 41
Historical reference analytes:       41
Matched analytes:                    41
Missing historical reference:        0

All current Duplicate analytes have a historical reference model.


In [8]:
# =============================================================================
# Interpolate historical reference values for current Duplicate observations
# =============================================================================

# Log-transform current mean concentration
current_dup_score_df["LOG_MEAN_CONC"] = np.log10(
    current_dup_score_df["MEAN_CONC"]
)

scored_tables = []

for analyte, current_group in current_dup_score_df.groupby("ANALYTE_CODE"):

    # Historical reference curve for this analyte
    ref_group = (
        reference_model_df[
            reference_model_df["ANALYTE_CODE"] == analyte
        ]
        .copy()
        .sort_values("LOG_MEAN_CONC")
    )

    # Current observations for this analyte
    scored_group = current_group.copy()

    x_ref = ref_group["LOG_MEAN_CONC"].to_numpy()
    x_current = scored_group["LOG_MEAN_CONC"].to_numpy()

    # ---------------------------------------------------------
    # Identify whether current observations fall within the
    # historical concentration range for this analyte
    # ---------------------------------------------------------

    ref_min = x_ref.min()
    ref_max = x_ref.max()

    scored_group["WITHIN_REFERENCE_RANGE"] = (
        (scored_group["LOG_MEAN_CONC"] >= ref_min)
        & (scored_group["LOG_MEAN_CONC"] <= ref_max)
    )

    # ---------------------------------------------------------
    # Interpolate historical quantiles and allowable RPD
    # ---------------------------------------------------------

    for col in ["Q25", "Q50", "Q75", "Q95", "ALLOWABLE_RPD"]:

        scored_group[f"REF_{col}"] = np.interp(
            x_current,
            x_ref,
            ref_group[col].to_numpy()
        )

    scored_tables.append(scored_group)

# ---------------------------------------------------------
# Combine all analytes
# ---------------------------------------------------------

current_dup_scored_df = pd.concat(
    scored_tables,
    ignore_index=True
)

# ---------------------------------------------------------
# Confirm scoring output
# ---------------------------------------------------------

print(
    f"Current Duplicate observations scored: "
    f"{len(current_dup_scored_df):,}"
)

print(
    f"Within historical concentration range: "
    f"{current_dup_scored_df['WITHIN_REFERENCE_RANGE'].sum():,}"
)

print(
    f"Outside historical concentration range: "
    f"{(~current_dup_scored_df['WITHIN_REFERENCE_RANGE']).sum():,}"
)

current_dup_scored_df[
    [
        "JOB_CODE",
        "ANALYTE_CODE",
        "MEAN_CONC",
        "RPD",
        "WITHIN_REFERENCE_RANGE",
        "REF_Q50",
        "REF_Q95",
        "REF_ALLOWABLE_RPD"
    ]
].head(20)


Current Duplicate observations scored: 4,437
Within historical concentration range: 4,430
Outside historical concentration range: 7


,JOB_CODE,ANALYTE_CODE,MEAN_CONC,RPD,WITHIN_REFERENCE_RANGE,REF_Q50,REF_Q95,REF_ALLOWABLE_RPD
0,TSV_LB0007353672,AG,0.188884,130.57,True,50.901692,402.489623,676.783710
1,TSV_LB0007353557,AG,0.012395,546.99,True,715.270657,1974.026926,10099.426840
2,TSV_LB0007353441,AG,0.288318,59.25,True,41.409058,219.455036,448.562686
3,TSV_LB0007353441,AG,0.386792,68.29,True,37.104288,129.296587,338.188823
4,TSV_LB0007353441,AG,0.279616,11.06,True,41.899700,230.765743,462.071534
5,TSV_LB0007341084,AG,0.053904,128.99,True,173.569402,1158.246446,2333.972090
6,TSV_LB0007341084,AG,0.888665,7.09,True,23.294288,35.234341,155.669775
7,TSV_LB0007340966,AG,0.184223,122.06,True,51.728106,414.959896,693.532741
8,TSV_LB0007340966,AG,0.330712,9.34,True,39.372343,173.144699,392.995561
9,TSV_LB0007340966,AG,0.246186,46.41,True,44.174100,281.188726,522.768834
